In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

In [ ]:
protein = "VCAM1"
synapse_type = "VGLUT1-PSD95"

In [ ]:
results_file = f"/Volumes/KINGSTON/data/phd/image-analysis/synapse-counting/{protein}/{protein}-LacZ_{synapse_type}_output_data/metric_results.csv"
results = pd.read_csv(results_file)
results.head(10)

In [ ]:
# Add a new column 'section' by extracting the "section-<number>" part from 'img_filename'
results['section'] = results['img_filename'].apply(lambda x: re.search(r'section-\d+', x).group(0) if re.search(r'section-\d+', x) else None)

# Select the relevant columns
results_mfi = results[['presynapse_image_mfi', 'gRNA', 'hippocampal_layer', 'section', 'Brain']]

results_mfi.head(10)

In [ ]:
scaler = StandardScaler()
results_mfi['presynapse_image_mfi_scaled'] = scaler.fit_transform(results_mfi[['presynapse_image_mfi']])

In [ ]:
results_mfi.head(10)

In [ ]:
# Create a combined key for the nested random effects as needed
results_mfi['Brain_gRNA'] = results_mfi['Brain'] + ':' + results_mfi['gRNA']
results_mfi['Brain_section_layer'] = results_mfi['Brain'] + ':' + results_mfi['section'] + ':' + results_mfi['hippocampal_layer']

In [ ]:
results_mfi.head(10)

In [ ]:
# for decimals
pd.set_option('display.precision', 7)

# Define the mixed-effects model
model = smf.mixedlm(
    "presynapse_image_mfi_scaled ~ gRNA * hippocampal_layer",   # Fixed effects formula
    data=results_mfi,
    groups="Brain",                                            # Random intercept for Brain
    re_formula="~gRNA"                                           # Include random intercept
)

# Fit the model using REML
model_results = model.fit(reml=True)

# Print the summary of the model
print(model_results.summary())

# To get the variance components
print("\nVariance Components:")
print(model_results.cov_re)

In [ ]:
# Get the summary of the results
summary = model_results.summary()

# Extract coefficients and p-values into a DataFrame
coefficients = model_results.params
p_values = model_results.pvalues

adj_pvals = multipletests(p_values, method='fdr_bh')[1]
significance = ['Significant' if p < 0.05 else 'Not Significant' for p in adj_pvals]


# Create a DataFrame for coefficients and p-values
coefficients_table = pd.DataFrame({
    'Variable': coefficients.index,
    'Coefficient': coefficients,
    'P > z': p_values,
    "adj_pvals": adj_pvals,
    "significance": significance
})

# Reset index for better formatting (optional)
coefficients_table.reset_index(drop=True, inplace=True)

# Display the coefficients table
coefficients_table.head(20)